In [0]:
df_bronze = spark.table("prf_acidentes.bronze.acidentes_raw")

# 1. Verificar duplicidade de ID (chave do acidente)
total_linhas = df_bronze.count()
total_ids_distintos = df_bronze.select("id").distinct().count()

print(f"Total de linhas: {total_linhas}")
print(f"Total de IDs distintos: {total_ids_distintos}")
print(f"Diferença (possíveis duplicatas): {total_linhas - total_ids_distintos}")

In [0]:
from pyspark.sql import functions as F

# Identifica quais colunas são string (evita comparar "" em colunas não-string)
colunas_string = [c for c, t in df_bronze.dtypes if t == "string"]
colunas_outras = [c for c, t in df_bronze.dtypes if t != "string"]

# Nulos/vazios só nas colunas string
nulos_string = df_bronze.select([
    F.sum(F.when(F.col(c).isNull() | (F.col(c) == ""), 1).otherwise(0)).alias(c)
    for c in colunas_string
])

# Nulos nas colunas não-string (sem comparar com "")
nulos_outras = df_bronze.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in colunas_outras
])

display(nulos_string)
display(nulos_outras)

In [0]:
# 3. Ver exemplos de valores problemáticos em campos numéricos-chave
display(df_bronze.select("km", "latitude", "longitude").distinct().limit(20))

In [0]:
from pyspark.sql import functions as F

df_silver = (
    df_bronze
    # Conversão de tipos numéricos simples
    .withColumn("id", F.expr("try_cast(try_cast(id AS DOUBLE) AS LONG)"))  # <-- ALTERADO AQUI
    .withColumn("br", F.col("br").cast("int"))
    .withColumn("pessoas", F.col("pessoas").cast("int"))
    .withColumn("mortos", F.col("mortos").cast("int"))
    .withColumn("feridos_leves", F.col("feridos_leves").cast("int"))
    .withColumn("feridos_graves", F.col("feridos_graves").cast("int"))
    .withColumn("ilesos", F.col("ilesos").cast("int"))
    .withColumn("ignorados", F.col("ignorados").cast("int"))
    .withColumn("feridos", F.col("feridos").cast("int"))
    .withColumn("veiculos", F.col("veiculos").cast("int"))
    # Correção de vírgula decimal -> ponto, depois cast para double
    .withColumn("km", F.regexp_replace(F.col("km"), ",", ".").cast("double"))
    .withColumn("latitude", F.regexp_replace(F.col("latitude"), ",", ".").cast("double"))
    .withColumn("longitude", F.regexp_replace(F.col("longitude"), ",", ".").cast("double"))
    # Data e hora
    .withColumn("data_inversa", F.to_date(F.col("data_inversa"), "yyyy-MM-dd"))
    .withColumn("hora", F.split(F.col("horario"), ":").getItem(0).cast("int"))
    # Padronização de texto: remove espaços extras nas colunas categóricas
    .withColumn("uf", F.trim(F.upper(F.col("uf"))))
    .withColumn("municipio", F.trim(F.col("municipio")))
    .withColumn("causa_acidente", F.trim(F.col("causa_acidente")))
    .withColumn("tipo_acidente", F.trim(F.col("tipo_acidente")))
    .withColumn("classificacao_acidente", F.trim(F.col("classificacao_acidente")))
    .withColumn("fase_dia", F.trim(F.col("fase_dia")))
    .withColumn("condicao_metereologica", F.trim(F.col("condicao_metereologica")))
    .withColumn("tipo_pista", F.trim(F.col("tipo_pista")))
    .withColumn("tracado_via", F.trim(F.col("tracado_via")))
    .withColumn("uso_solo", F.trim(F.col("uso_solo")))
    .withColumn("sentido_via", F.trim(F.col("sentido_via")))
    # Flag de fim de semana
    .withColumn("dia_semana", F.trim(F.col("dia_semana")))
    .withColumn("flag_fim_de_semana", 
        F.when(F.lower(F.col("dia_semana")).isin("sábado", "sabado", "domingo"), True)
         .otherwise(False))
)

df_silver.printSchema()
print("Total de linhas:", df_silver.count())

In [0]:
df_silver.select(F.sum(F.when(F.col("id").isNull(), 1).otherwise(0)).alias("id_nulos_apos_cast")).show()

In [0]:
from pyspark.sql import functions as F

df_silver.select(
    F.min("data_inversa").alias("data_min"),
    F.max("data_inversa").alias("data_max"),
    F.min("km").alias("km_min"),
    F.max("km").alias("km_max"),
    F.min("latitude").alias("lat_min"),
    F.max("latitude").alias("lat_max"),
    F.min("hora").alias("hora_min"),
    F.max("hora").alias("hora_max"),
    F.sum(F.when(F.col("km").isNull(), 1).otherwise(0)).alias("km_nulos_apos_cast"),
    F.sum(F.when(F.col("latitude").isNull(), 1).otherwise(0)).alias("lat_nulos_apos_cast")
).show()

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("prf_acidentes.silver.acidentes_limpo")

print("Tabela Silver criada com sucesso!")

In [0]:
display(spark.sql("SELECT * FROM prf_acidentes.silver.acidentes_limpo LIMIT 10"))